<a href="https://colab.research.google.com/github/tema-coder/qwen_laro_adapt/blob/main/nb/Qwen3_5_(4B)_Vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

You can now train embedding models 1.8-3.3x faster with 20% less VRAM. [Blog](https://unsloth.ai/docs/new/embedding-finetuning)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

3x faster LLM training with 30% less VRAM and 500K context. [3x faster](https://unsloth.ai/docs/new/3x-faster-training-packing) • [500K Context](https://unsloth.ai/docs/new/500k-context-length-fine-tuning)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==5.3.0
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit", # Llama 3.2 vision support
    "unsloth/Llama-3.2-11B-Vision-bnb-4bit",
    "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit", # Can fit in a 80GB card!
    "unsloth/Llama-3.2-90B-Vision-bnb-4bit",

    "unsloth/Pixtral-12B-2409-bnb-4bit",              # Pixtral fits in 16GB!
    "unsloth/Pixtral-12B-Base-2409-bnb-4bit",         # Pixtral base model

    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",          # Qwen2 VL support
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    "unsloth/Qwen2-VL-72B-Instruct-bnb-4bit",

    "unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit",      # Any Llava variant works!
    "unsloth/llava-1.5-7b-hf-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",
    load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

<font size="6"> !!! Убираем True у параметра finetune_vision_layers !!!   </font>

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

<a name="Data"></a>
### Data Prep
We'll be using a sampled dataset of handwritten maths formulas. The goal is to convert these images into a computer readable form - ie in LaTeX form, so we can render it. This can be very useful for complex formulas.

You can access the dataset [here](https://huggingface.co/datasets/unsloth/LaTeX_OCR). The full dataset is [here](https://huggingface.co/datasets/linxy/LaTeX_OCR).

   <font size="6">!!! обрабатываем данные из excel файла !!! </font>

In [ ]:
import pandas as pd
from datasets import Dataset

# 1. Загружаем твой Excel файл
# Убедись, что файл train_dataset.xlsx загружен в папку с ноутбуком (в Colab слева значок папки)
df = pd.read_excel("train_dataset.xlsx", sheet_name="Sheet1")

# 2. Функция для превращения строки таблицы в структуру диалога
def convert_to_conversation(row):
    # Мы берем текст пользователя и ответ модели
    user_text = row["text"]
    assistant_text = row["answer"]

    # Возвращаем структуру, которую ждет Unsloth/Qwen
    return {
        "conversations": [
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text}
        ]
    }

# 3. Применяем функцию ко всем строкам таблицы
# Это превратит DataFrame в список словарей
dataset_converted = [convert_to_conversation(row) for index, row in df.iterrows()]

# 4. Превращаем список в Dataset, понятный Hugging Face
dataset = Dataset.from_list(dataset_converted)


Let's take an overview look at the dataset. We shall see what the 3rd image is, and what caption it had.

In [ ]:
dataset

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
 print(dataset[0]["text"])

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!

We use our new `UnslothVisionDataCollator` which will help in our vision finetuning setup.

In [ ]:
from trl import SFTTrainer, SFTConfig
# UnslothVisionDataCollator НЕ нужен, так как нет картинок!

FastVisionModel.for_training(model) # Включаем режим обучения

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,  # Используем ваш датасет с колонкой "text"
    eval_dataset = None,      # Можно добавить валидацию позже
    args = SFTConfig(
        # --- Пути и сохранение ---
        output_dir = "/content/drive/MyDrive/qwen_3_5_lora_checkpoint",
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 2,
        logging_steps = 1,
        report_to = "none",

        # --- Параметры обучения ---
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Эффективный батч = 2 * 4 = 8
        warmup_steps = 5,
        num_train_epochs = 1,            # 1 полная эпоха (лучше чем max_steps)
        learning_rate = 2e-4,
        optim = "adamw_8bit",            # Оптимизатор Unsloth
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,

        # --- Формат данных ---
        dataset_text_field = "text",     # Колонка, созданная в formatting_prompts_func
        remove_unused_columns = True,    # Можно включить, лишние колонки удалятся
        max_length = 2048,               # Максимальная длина контекста
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [ ]:
# ==========================================
# БЛОК РУЧНОЙ ПРОВЕРКИ МОДЕЛИ
# ==========================================

# 1. Переключаем модель в режим инференса (ОБЯЗАТЕЛЬНО после обучения!)
FastVisionModel.for_inference(model)

# 2. Напиши сюда промт из одной из твоих 6 отложенных строк
my_test_prompt = "Напиши сюда свой промт из Excel..."

# 3. Формируем структуру диалога
messages = [
    {"role": "user", "content": my_test_prompt}
]

# 4. Применяем шаблон Qwen
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Обязательно True, чтобы модель начала отвечать
)

# 5. Запускаем генерацию
from transformers import TextStreamer
print(f"ЗАПРОС: {my_test_prompt}\n")
print("ОТВЕТ МОДЕЛИ:")
print("-" * 50)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 512,   # Сколько токенов генерировать (можно уменьшить для скорости)
    temperature = 0.7,      # Креативность (0.7 - сбалансировано)
    top_p = 0.8,            # Фильтр вероятности
    top_k = 20,             # Фильтр топ-токенов
    do_sample = True,       # Включаем сэмплирование (иначе temperature не работает)
    pad_token_id = tokenizer.pad_token_id, # Предупреждение о pad_token
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

print("-" * 50)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("/content/drive/MyDrive/qwen_lora_3_5_final")
tokenizer.save_pretrained("/content/drive/MyDrive/qwen_lora_3_5_final")

  <font size="6">!!! Проверка модели после сохранения !!! </font>

  <font size="6">!!! 1) Загружаем БАЗОВУЮ модель
                      2) Настраиваем архитектуру LoRA 3)Загружаем веса адаптера с Диска !!! </font>

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==5.3.0
!pip install --no-deps trl==0.22.2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from unsloth import FastVisionModel
import torch

# --- 1. Загружаем БАЗОВУЮ модель ---
# Она должна быть точно такой же, как та, на которой вы обучались!
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",  # Идентификатор базовой модели
    load_in_4bit = False,  # Ставьте True, если не хватает памяти (как при обучении)
    use_gradient_checkpointing = "unsloth",
)

# --- 2. Настраиваем архитектуру LoRA ---
# Параметры должны СОВПАДАТЬ с настройками при обучении!
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers = False,  # Должно совпадать с обучением
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16,              # Должно совпадать с обучением
    lora_alpha = 16,     # Должно совпадать с обучением
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

# --- 3. Загружаем ваши веса адаптера с Диска ---
# Укажите путь к папке, куда вы сохранили модель (где лежат adapter_model.safetensors)
adapter_path = "/content/drive/MyDrive/qwen_lora_3_5_final"
model.load_adapter(adapter_path, adapter_name="default")

# --- 4. Переключаем в режим инференса ---
FastVisionModel.for_inference(model)

print("✅ Модель готова к работе!")

In [ ]:
from transformers import TextStreamer

# Ваш тестовый промпт
my_test_prompt = "Ваш вопрос из отложенной выборки..."

messages = [{"role": "user", "content": my_test_prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
)

print(f"ЗАПРОС: {my_test_prompt}\n")
print("ОТВЕТ МОДЕЛИ:")

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 512,
    temperature = 0.7,
    top_p = 0.8,
    top_k = 20,
    do_sample = True,
    pad_token_id = tokenizer.pad_token_id,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)